# 01 — Render-Trained Unified Matte + Depth + Normal Student (E1 + E2)

Implements `internal/plans/2d-view/render-trained/PLAN.md`:
- **Backbone:** `mobilenetv4_conv_small` / `repvit_m0_9` (~10–15 MB FP16 ONNX, <= 90 ms M2 CPU, < 100 MB RSS).
- **Data:** Infinite `tools/render-trained/make_data.py` (2x supersampled `composite`, `alpha`, `depth`, `normal`) + PD12M CC0 with SAM 2.1 Apache-2.0 pseudo-masks.
- **Replaces:** `u2netp` (0.690), `MODNet` (0.702, 158 ms), `depth.onnx` (94 MB), and `MoGe-2` (70 MB, academic data question).


In [ ]:
!pip -q uninstall -y torchaudio
!pip -q install -U torch torchvision timm onnx onnxscript onnxslim onnxruntime datasets transformers pillow numpy tqdm
import os, hashlib, torch, onnx, onnxslim, onnxruntime as ort
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/out', exist_ok=True)
!unzip -q -o /content/drive/MyDrive/infinite_ml/e1_200.zip -d /content/
DATA_DIR = '/content/e1_200'
print("Scenes:", len(os.listdir(DATA_DIR)))


In [ ]:
import torch.nn as nn, torch.nn.functional as F, timm
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np, glob

class RenderTrainedDataset(Dataset):
    def __init__(self, root, size=512):
        self.dirs = sorted(glob.glob(os.path.join(root, 'scene_*')))
        self.size = size
    def __len__(self): return len(self.dirs)
    def __getitem__(self, i):
        d = self.dirs[i]
        load_rgb = lambda f: torch.from_numpy(np.asarray(Image.open(os.path.join(d, f)).convert('RGB').resize((self.size, self.size)), np.float32) / 255.0).permute(2,0,1)
        load_l   = lambda f: torch.from_numpy(np.asarray(Image.open(os.path.join(d, f)).convert('L').resize((self.size, self.size)), np.float32) / 255.0)[None]
        return load_rgb('composite.png'), load_l('alpha.png'), load_l('depth.png'), load_rgb('normal.png') * 2.0 - 1.0

class UnifiedStudent(nn.Module):
    def __init__(self, backbone='mobilenetv4_conv_small.e2400_r224_in1k'):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=True, features_only=True, out_indices=(1,2,3,4))
        chs = self.enc.feature_info.channels()
        self.lat = nn.ModuleList([nn.Conv2d(c, 64, 1) for c in chs])
        self.fuse = nn.Sequential(nn.Conv2d(64*4, 96, 3, padding=1), nn.ReLU(inplace=True),
                                  nn.Conv2d(96, 64, 3, padding=1), nn.ReLU(inplace=True))
        self.matte_head  = nn.Conv2d(64, 1, 1)
        self.depth_head  = nn.Conv2d(64, 1, 1)
        self.normal_head = nn.Conv2d(64, 3, 1)

    def forward(self, x):
        H, W = x.shape[2], x.shape[3]
        feats = [F.interpolate(l(f), size=(H//4, W//4), mode='bilinear', align_corners=False)
                 for l, f in zip(self.lat, self.enc(x))]
        h = F.interpolate(self.fuse(torch.cat(feats, dim=1)), size=(H, W), mode='bilinear', align_corners=False)
        alpha = torch.sigmoid(self.matte_head(h))
        depth = torch.sigmoid(self.depth_head(h))
        norm  = F.normalize(self.normal_head(h), dim=1)
        return alpha, depth, norm

def train_e1_e2(epochs=20, lr=3e-4, batch_size=8):
    ds = RenderTrainedDataset(DATA_DIR)
    assert len(ds) > 0, f"No scene_* folders found in {DATA_DIR}; run tools/render-trained/make_data.py first."
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=2)
    model = UnifiedStudent().cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    for ep in range(epochs):
        tot = 0.0
        for img, gt_a, gt_d, gt_n in dl:
            img, gt_a, gt_d, gt_n = img.cuda(), gt_a.cuda(), gt_d.cuda(), gt_n.cuda()
            pa, pd, pn = model(img)
            loss_a = F.l1_loss(pa, gt_a) + F.l1_loss(pa[:,:,1:]-pa[:,:,:-1], gt_a[:,:,1:]-gt_a[:,:,:-1])
            loss_d = F.l1_loss(pd, gt_d)
            loss_n = (1.0 - (pn * gt_n).sum(dim=1)).mean()
            loss = loss_a + 0.5 * loss_d + 0.5 * loss_n
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()
        print(f"Epoch {ep+1}/{epochs} loss={tot/len(dl):.4f}")
    return model


In [ ]:
# Trajectory B Run 2: BCE+Dice matte loss (prevents zero-collapse) + filtered PD12M (0.08 <= mean(a) <= 0.85) + fg-masked normal loss
import glob, random, shutil, urllib.request
from io import BytesIO
from datasets import load_dataset
from transformers import pipeline as hf_pipeline

pd_dir = '/content/pd12m_400'
os.makedirs(pd_dir, exist_ok=True)
if len(glob.glob(f'{pd_dir}/*.jpg')) < 350:
    ds_stream = load_dataset('Spawning/PD12M', split='train', streaming=True)
    saved = 0
    for row in ds_stream:
        if saved >= 400: break
        url = row.get('url') or row.get('image_url') or ''
        if not url.startswith('http'): continue
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'infinite-ml'})
            with urllib.request.urlopen(req, timeout=5) as r:
                im = Image.open(BytesIO(r.read())).convert('RGB')
            if min(im.size) < 256: continue
            im.resize((512, 512), Image.BILINEAR).save(f'{pd_dir}/pd_{saved:04d}.jpg', quality=92)
            saved += 1
        except Exception:
            continue
    print('Fetched PD12M CC0 photos:', saved)

modnet_path = '/content/modnet_fp16.onnx'
if not os.path.exists(modnet_path):
    urllib.request.urlretrieve('https://huggingface.co/Xenova/modnet/resolve/main/onnx/model_fp16.onnx', modnet_path)
mod_sess = ort.InferenceSession(modnet_path, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
depth_pipe = hf_pipeline('depth-estimation', model='depth-anything/Depth-Anything-V2-Small-hf', device=0)

pd_lbl = '/content/pd12m_lbl_v2'
os.makedirs(pd_lbl, exist_ok=True)
real_items = []
for p in sorted(glob.glob(f'{pd_dir}/*.jpg')):
    stem = os.path.splitext(os.path.basename(p))[0]
    ap, dp = f'{pd_lbl}/{stem}_a.png', f'{pd_lbl}/{stem}_d.png'
    im = Image.open(p).convert('RGB')
    if not os.path.exists(ap):
        x = (np.asarray(im, np.float32) / 255.0 - 0.5) / 0.5
        a = mod_sess.run(None, {'input': x.transpose(2,0,1)[None].astype(np.float32)})[0][0,0]
        Image.fromarray(np.clip(a * 255, 0, 255).astype(np.uint8)).save(ap)
    else:
        a = np.asarray(Image.open(ap).convert('L'), np.float32) / 255.0
    # Filter out all-zero / full-frame masks so the student never collapses!
    if not (0.08 <= float(a.mean()) <= 0.85):
        continue
    if not os.path.exists(dp):
        depth_pipe(im)['depth'].resize((512, 512), Image.BILINEAR).save(dp)
    real_items.append((p, ap, dp))
print(f'Kept non-empty real PD12M teacher pairs: {len(real_items)} / 400')

bg_pool = sorted(glob.glob(f'{pd_dir}/*.jpg'))
class Run2Dataset(Dataset):
    def __init__(self, rroot, reals, bgs, size=512):
        self.renders = sorted(glob.glob(os.path.join(rroot, 'scene_*')))
        self.reals, self.bgs, self.size = reals, bgs, size
    def __len__(self): return len(self.renders) + len(self.reals) * 2
    def __getitem__(self, i):
        if i < len(self.renders):
            d = self.renders[i]
            comp = np.asarray(Image.open(os.path.join(d, 'composite.png')).convert('RGB').resize((self.size, self.size)), np.float32) / 255.0
            a = np.asarray(Image.open(os.path.join(d, 'alpha.png')).convert('L').resize((self.size, self.size)), np.float32) / 255.0
            dep = np.asarray(Image.open(os.path.join(d, 'depth.png')).convert('L').resize((self.size, self.size)), np.float32) / 255.0
            nrm = np.asarray(Image.open(os.path.join(d, 'normal.png')).convert('RGB').resize((self.size, self.size)), np.float32) / 255.0 * 2.0 - 1.0
            if self.bgs and random.random() < 0.7:
                bg = np.asarray(Image.open(random.choice(self.bgs)).convert('RGB').resize((self.size, self.size)), np.float32) / 255.0
                comp = comp * a[:,:,None] + bg * (1.0 - a[:,:,None])
            return torch.from_numpy(comp).permute(2,0,1), torch.from_numpy(a)[None], torch.from_numpy(dep)[None], torch.from_numpy(nrm).permute(2,0,1), torch.tensor(1.0)
        p, ap, dp = self.reals[(i - len(self.renders)) % len(self.reals)]
        comp = np.asarray(Image.open(p).convert('RGB').resize((self.size, self.size)), np.float32) / 255.0
        a = np.asarray(Image.open(ap).convert('L').resize((self.size, self.size)), np.float32) / 255.0
        dep = np.asarray(Image.open(dp).convert('L').resize((self.size, self.size)), np.float32) / 255.0
        return torch.from_numpy(comp).permute(2,0,1), torch.from_numpy(a)[None], torch.from_numpy(dep)[None], torch.zeros(3, self.size, self.size), torch.tensor(0.0)

dl2 = DataLoader(Run2Dataset('/content/e1_200', real_items, bg_pool), batch_size=8, shuffle=True, num_workers=2)
model_v2 = UnifiedStudent().cuda()
opt = torch.optim.AdamW(model_v2.parameters(), lr=5e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=25)

for ep in range(25):
    tot = 0.0
    for img, gt_a, gt_d, gt_n, has_n in dl2:
        img, gt_a, gt_d, gt_n, has_n = img.cuda(), gt_a.cuda(), gt_d.cuda(), gt_n.cuda(), has_n.cuda()
        pa, pd, pn = model_v2(img)
        bce = F.binary_cross_entropy(pa.clamp(1e-5, 1 - 1e-5), gt_a)
        inter = (pa * gt_a).sum(dim=(1,2,3))
        dice = 1.0 - (2.0 * inter + 1.0) / (pa.sum(dim=(1,2,3)) + gt_a.sum(dim=(1,2,3)) + 1.0)
        loss_a = bce + dice.mean() + 0.5 * F.l1_loss(pa, gt_a)
        loss_d = F.l1_loss(pd, gt_d)
        fg_mask = (gt_a[:,0] > 0.5).float() * has_n[:, None, None]
        cos_err = (1.0 - (pn * F.normalize(gt_n, dim=1)).sum(dim=1)) * fg_mask
        loss_n = cos_err.sum() / (fg_mask.sum() + 1e-6)
        loss = loss_a + 0.5 * loss_d + 1.5 * loss_n
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()
    sched.step()
    if (ep + 1) in (1, 5, 10, 15, 20, 25):
        print(f"Run2 Epoch {ep+1}/25 loss={tot/len(dl2):.4f}")

out_p = '/content/out/render_matte_depth_normal_v2.onnx'
torch.onnx.export(model_v2.eval().cpu(), torch.randn(1,3,512,512), out_p, opset_version=17, dynamo=False,
                  input_names=['image'], output_names=['alpha', 'depth', 'normal'])
onnx.checker.check_model(out_p)
data = open(out_p, 'rb').read()
sha = hashlib.sha256(data).hexdigest()
shutil.copy2(out_p, '/content/drive/MyDrive/infinite_ml/render_matte_depth_normal_v2.onnx')
print(f"Run2 Exported {out_p}: {len(data)/1e6:.2f} MB sha256={sha}")
